[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/wolves_juliaslm.ipynb)

# Wolves Compression: SVD-based Weight Pruning for JuliaSLM

Uses evolutionary selection ("wolves") to find optimal per-layer SVD rank
thresholds for compressing JuliaSLM (5M params, val_loss=3.54). Each "wolf" is a rank
schedule evaluated by `fitness = -(val_loss + beta * weight_entropy)`. Tournament
selection converges on the best compression strategy.

**Target model**: [JuliaSLM](https://huggingface.co/LisaMegaWatts/JuliaSLM) — LLaMA-style decoder with standard MHA (4 heads), RMSNorm, SwiGLU, RoPE, weight-tied output. ~5M params, d=256, 6 layers, val_loss=3.54 (curated philosophy corpus).

**Why JuliaSLM instead of JuliaFluxGPT?** The first wolves run on JuliaFluxGPT-23M (dirty data, val_loss=6.62) produced a flat fitness landscape — 0.007 loss improvement over 5 gens. Dirty weights lack SVD structure. JuliaSLM has well-trained weights from curated data, so SVD should find real compressibility. It's also 5× smaller = 5× faster evals.

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!mkdir -p /content/SymbioGPT
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. W&B + HF login
import wandb
from huggingface_hub import login as hf_login

wandb.login()
hf_login()

In [ ]:
# 4. Load data + download JuliaSLM weights
import os, sys, math, time, copy, random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
import numpy as np
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Download pre-tokenized data (same 2000-token BPE vocab)
DATA_REPO = "LisaMegaWatts/SymbioGPT-10M"
SLM_REPO = "LisaMegaWatts/JuliaSLM"
os.makedirs("data", exist_ok=True)

print("Downloading pre-tokenized data...")
hf_hub_download(repo_id=DATA_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="data/val.txt.tokens.pt", local_dir=".")

print("Downloading JuliaSLM weights (NPZ)...")
hf_hub_download(repo_id=SLM_REPO, filename="juliaslm_weights.npz", local_dir=".")

# Chunk tokens into (input, target) pairs
CTX = 256
print("Loading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs")
del train_tokens, val_tokens

In [ ]:
# 5. JuliaSLM PyTorch model definition
#
# Standard LLaMA-style: MHA (4 heads, separate Q/K/V/O), RMSNorm, SwiGLU, RoPE.
# No GQA, no fused wkv — simpler than JuliaFluxGPT.
#
# IMPORTANT: Julia's column-major reshape(result, HD, T, H, B) fills dims
# HD→T→H→B (fastest→slowest). Python's row-major equivalent is
# view(B, H, T, HD) — NOT view(B, T, H, HD).transpose(1, 2).
# The model was TRAINED with Julia's layout, so we must match it exactly.

@dataclass
class JuliaSLMConfig:
    d_model: int = 256
    n_layers: int = 6
    n_heads: int = 4
    head_dim: int = 64
    ffn_inner: int = 640
    context_length: int = 256
    vocab_size: int = 2000
    weight_tying: bool = True
    rope_base: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 256, base: float = 10000.0):
        super().__init__()
        freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)
        self.register_buffer("cos_cache", angles.cos())
        self.register_buffer("sin_cache", angles.sin())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, n_heads, T, head_dim)
        seq_len = x.size(2)
        half = x.size(-1) // 2
        x1, x2 = x[..., :half], x[..., half:]
        cos = self.cos_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len, :half].unsqueeze(0).unsqueeze(0)
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class CausalSelfAttention(nn.Module):
    """Standard multi-head attention with separate Q, K, V, O projections."""

    def __init__(self, d_model: int, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        total_dim = n_heads * head_dim
        self.wq = nn.Linear(d_model, total_dim, bias=False)
        self.wk = nn.Linear(d_model, total_dim, bias=False)
        self.wv = nn.Linear(d_model, total_dim, bias=False)
        self.wo = nn.Linear(total_dim, d_model, bias=False)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        H, HD = self.n_heads, self.head_dim

        # Julia-matching reshape: view(B, H, T, HD) matches Julia's
        # column-major reshape(HD, T, H, B)
        q = self.wq(x).view(B, H, T, HD)
        k = self.wk(x).view(B, H, T, HD)
        v = self.wv(x).view(B, H, T, HD)

        q = rope(q)
        k = rope(k)

        scale = 1.0 / math.sqrt(HD)
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = attn + mask
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)

        out = out.contiguous().view(B, T, H * HD)
        return self.wo(out)


class SwiGLUFFN(nn.Module):
    """SwiGLU feed-forward: gate=silu(w1*x) * v*x, out=w2*gate."""

    def __init__(self, d_model: int, inner_dim: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, inner_dim, bias=False)  # gate
        self.v = nn.Linear(d_model, inner_dim, bias=False)   # up
        self.w2 = nn.Linear(inner_dim, d_model, bias=False)  # down

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.v(x))


class TransformerBlock(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.ln1 = RMSNorm(config.d_model)
        self.attn = CausalSelfAttention(config.d_model, config.n_heads, config.head_dim)
        self.ln2 = RMSNorm(config.d_model)
        self.ffn = SwiGLUFFN(config.d_model, config.ffn_inner)

    def forward(self, x: torch.Tensor, rope: RotaryEmbedding,
                mask: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x), rope, mask)
        x = x + self.ffn(self.ln2(x))
        return x


class JuliaSLM(nn.Module):
    def __init__(self, config: JuliaSLMConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.rope = RotaryEmbedding(config.head_dim, config.context_length, config.rope_base)
        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.ln_f = RMSNorm(config.d_model)
        # Weight-tied: no separate head

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        mask = torch.triu(
            torch.full((T, T), float("-inf"), device=x.device, dtype=x.dtype),
            diagonal=1,
        )
        for block in self.blocks:
            x = block(x, self.rope, mask)
        x = self.ln_f(x)
        return F.linear(x, self.tok_emb.weight)  # weight-tied output


config = JuliaSLMConfig()
print(f"JuliaSLM config: d={config.d_model}, L={config.n_layers}, H={config.n_heads}, "
      f"hd={config.head_dim}, ffn={config.ffn_inner}, vocab={config.vocab_size}")

In [ ]:
# 6. Load JuliaSLM weights from NPZ

npz_data = np.load("juliaslm_weights.npz")

# Print NPZ structure
print("NPZ keys:")
for key in sorted(npz_data.files):
    arr = npz_data[key]
    print(f"  {key}: {arr.shape} {arr.dtype}")

# Read hyperparams from NPZ
hp = {k: int(npz_data[k][0]) for k in npz_data.files if k.startswith("_hp_")}
print(f"\nHyperparams: {hp}")

# Build model
model = JuliaSLM(config).to(device)

# Map NPZ keys to PyTorch state_dict keys
# NPZ stores bare matrices (e.g. "blocks.0.attn.wq") but nn.Linear
# expects ".weight" suffix (e.g. "blocks.0.attn.wq.weight").
# Embedding, RMSNorm, and ln_f already have ".weight" in the key.
model_keys = set(model.state_dict().keys())
state_dict = {}
for key in npz_data.files:
    if key.startswith("_hp_"):
        continue
    tensor = torch.from_numpy(npz_data[key].copy())
    if key in model_keys:
        # Direct match (tok_emb.weight, ln1.weight, ln2.weight, ln_f.weight)
        state_dict[key] = tensor
    elif key + ".weight" in model_keys:
        # nn.Linear: NPZ "blocks.0.attn.wq" → PyTorch "blocks.0.attn.wq.weight"
        state_dict[key + ".weight"] = tensor
    else:
        print(f"  WARNING: no match for NPZ key '{key}'")

# Load weights (strict=False for RoPE buffers which are computed, not loaded)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"\nLoaded weights: {len(state_dict)} arrays into model")
if missing:
    # Only RoPE buffers should be missing
    rope_missing = [k for k in missing if "rope" in k or "cache" in k]
    other_missing = [k for k in missing if k not in rope_missing]
    if rope_missing:
        print(f"Missing (expected, RoPE buffers): {rope_missing}")
    if other_missing:
        print(f"WARNING — Missing weights: {other_missing}")
if unexpected:
    print(f"Unexpected: {unexpected}")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {n_params:,} params ({n_params/1e6:.2f}M)")

# Verify forward pass
with torch.no_grad():
    test_x = torch.randint(0, config.vocab_size, (1, 32), device=device)
    test_logits = model(test_x)
    print(f"Forward pass OK: {test_x.shape} -> {test_logits.shape}")
    print(f"Logits range: [{test_logits.min().item():.3f}, {test_logits.max().item():.3f}]")

In [ ]:
# 7. Baseline evaluation + compression metrics

def evaluate_model(model, val_inputs, val_labels, batch_size=64):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(dev)
            batch_tgt = val_labels[i:i+batch_size].to(dev)
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


def weight_entropy(model):
    """Shannon entropy of weight distribution (bits), 100 bins."""
    all_w = torch.cat([p.detach().cpu().flatten() for p in model.parameters()])
    if all_w.numel() == 0:
        return 0.0
    hist = torch.histc(all_w.float(), bins=100)
    probs = hist / hist.sum()
    probs = probs[probs > 0]
    return -(probs * torch.log2(probs)).sum().item()


def effective_rank(model):
    """Average effective rank across all Linear layers (SVD, >1% threshold)."""
    ranks = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            w = module.weight.detach().cpu()
            s = torch.linalg.svdvals(w)
            threshold = 0.01 * s[0] if s.numel() > 0 and s[0] > 0 else 0.0
            ranks.append((s > threshold).sum().item())
    return sum(ranks) / len(ranks) if ranks else 0.0


# Baseline
print("Computing baseline for JuliaSLM (~5M params)...")
base_loss, base_ppl = evaluate_model(model, val_inputs, val_labels)
base_entropy = weight_entropy(model)
base_eff_rank = effective_rank(model)

print(f"\n{'='*50}")
print(f"BASELINE: JuliaSLM (~5M)")
print(f"{'='*50}")
print(f"  Params:          {n_params:,}")
print(f"  Val Loss:        {base_loss:.4f}")
print(f"  Val PPL:         {base_ppl:.1f}")
print(f"  Weight Entropy:  {base_entropy:.4f} bits")
print(f"  Effective Rank:  {base_eff_rank:.1f}")
print(f"  Known val_loss:  3.5403 (from Julia training step 12305)")

In [ ]:
# 8. SVD Diagnostics — per-layer analysis

def diagnose_layers(model):
    """Analyze each Linear layer's SVD spectrum."""
    results = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        w = module.weight.detach().cpu()
        out_dim, in_dim = w.shape
        full_rank = min(out_dim, in_dim)
        n_params = out_dim * in_dim

        s = torch.linalg.svdvals(w)
        threshold = 0.01 * s[0]
        eff_rank = (s > threshold).sum().item()

        total_energy = (s ** 2).sum().item()
        cum_energy = torch.cumsum(s ** 2, dim=0)

        rank_90 = (cum_energy < 0.90 * total_energy).sum().item() + 1
        rank_95 = (cum_energy < 0.95 * total_energy).sum().item() + 1
        rank_99 = (cum_energy < 0.99 * total_energy).sum().item() + 1

        results.append({
            "name": name,
            "shape": (out_dim, in_dim),
            "full_rank": full_rank,
            "eff_rank": int(eff_rank),
            "rank_90": int(rank_90),
            "rank_95": int(rank_95),
            "rank_99": int(rank_99),
            "params": n_params,
            "compress_ratio": eff_rank / full_rank,
        })
    return results


diag = diagnose_layers(model)

print(f"{'Layer':<40} {'Shape':>12} {'Full':>5} {'Eff':>5} {'R90':>5} {'R95':>5} {'R99':>5} {'Ratio':>6}")
print("-" * 90)
total_params_linear = 0
total_params_at_r95 = 0
for d in diag:
    out_d, in_d = d["shape"]
    params_at_r95 = d["rank_95"] * (out_d + in_d)
    total_params_linear += d["params"]
    total_params_at_r95 += params_at_r95
    print(f"{d['name']:<40} {str(d['shape']):>12} {d['full_rank']:>5} {d['eff_rank']:>5} "
          f"{d['rank_90']:>5} {d['rank_95']:>5} {d['rank_99']:>5} {d['compress_ratio']:>6.2f}")

print(f"\nTotal Linear params: {total_params_linear:,}")
print(f"Params at rank-95:   {total_params_at_r95:,}")
print(f"Potential savings:   {total_params_linear - total_params_at_r95:,} "
      f"({(1 - total_params_at_r95/total_params_linear)*100:.1f}%)")

In [ ]:
# 9. FactoredLinear + compress_model()

class FactoredLinear(nn.Module):
    """SVD-compressed linear: W ≈ A @ B where A=(out,k), B=(k,in).

    Params: k * (out + in) instead of out * in.
    """

    def __init__(self, A: torch.Tensor, B: torch.Tensor):
        super().__init__()
        self.A = nn.Parameter(A)  # (out, k)
        self.B = nn.Parameter(B)  # (k, in)

    @property
    def weight(self):
        return self.A @ self.B

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(F.linear(x, self.B), self.A)


def compress_layer(linear: nn.Linear, target_rank: int) -> nn.Module:
    """Replace nn.Linear with FactoredLinear via SVD truncation."""
    w = linear.weight.detach()
    full_rank = min(w.shape)
    k = min(target_rank, full_rank)
    if k >= full_rank:
        return linear

    U, S, Vh = torch.linalg.svd(w, full_matrices=False)
    sqrt_S = S[:k].sqrt()
    A = U[:, :k] * sqrt_S.unsqueeze(0)    # (out, k)
    B = sqrt_S.unsqueeze(1) * Vh[:k, :]    # (k, in)
    return FactoredLinear(A, B)


def get_linear_layer_names(model):
    """Get names of all nn.Linear modules."""
    return [name for name, m in model.named_modules() if isinstance(m, nn.Linear)]


def compress_model(model, rank_schedule: Dict[str, int]):
    """Create compressed copy with per-layer SVD rank targets."""
    compressed = copy.deepcopy(model)
    for name, target_rank in rank_schedule.items():
        parts = name.split(".")
        parent = compressed
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr = parts[-1]
        linear = getattr(parent, attr)
        if isinstance(linear, nn.Linear):
            factored = compress_layer(linear, target_rank)
            setattr(parent, attr, factored)
    return compressed


# Quick test
layer_names = get_linear_layer_names(model)
print(f"Found {len(layer_names)} Linear layers:")
for name in layer_names:
    print(f"  {name}")

# Test compress on one layer
test_schedule = {layer_names[0]: 64}
test_compressed = compress_model(model, test_schedule)
test_params = sum(p.numel() for p in test_compressed.parameters())
print(f"\nOriginal params: {n_params:,}")
print(f"After 1 layer at rank 64: {test_params:,} ({n_params - test_params:,} saved)")
del test_compressed

In [ ]:
# 10. Wolf dataclass + population helpers

@dataclass
class Wolf:
    """A compression candidate: a rank schedule + its fitness."""
    rank_schedule: Dict[str, int]
    val_loss: float = float("inf")
    weight_ent: float = 0.0
    n_params: int = 0
    fitness: float = float("-inf")

    def summary(self):
        ranks = list(self.rank_schedule.values())
        return (f"params={self.n_params:,} loss={self.val_loss:.4f} "
                f"ent={self.weight_ent:.3f} fit={self.fitness:.4f} "
                f"ranks=[{min(ranks)}-{max(ranks)}]")


def evaluate_wolf(wolf: Wolf, base_model, val_inputs, val_labels,
                  beta: float = 0.01) -> Wolf:
    """Compress model per wolf's schedule, evaluate, compute fitness."""
    compressed = compress_model(base_model, wolf.rank_schedule)
    compressed.to(device)
    wolf.val_loss, _ = evaluate_model(compressed, val_inputs, val_labels)
    wolf.weight_ent = weight_entropy(compressed)
    wolf.n_params = sum(p.numel() for p in compressed.parameters())
    wolf.fitness = -(wolf.val_loss + beta * wolf.weight_ent)
    del compressed
    torch.cuda.empty_cache()
    return wolf


def random_rank_schedule(layer_names, diag_info, min_frac=0.15, max_frac=0.85):
    """Generate random rank schedule between min_frac and max_frac of effective rank."""
    schedule = {}
    diag_map = {d["name"]: d for d in diag_info}
    for name in layer_names:
        d = diag_map[name]
        eff = d["eff_rank"]
        low = max(4, int(eff * min_frac))
        high = max(low + 1, int(eff * max_frac))
        schedule[name] = random.randint(low, high)
    return schedule


def tournament_select(population: List[Wolf], k: int = 3) -> Wolf:
    """Select best wolf from random tournament of size k."""
    contestants = random.sample(population, min(k, len(population)))
    return max(contestants, key=lambda w: w.fitness)


def crossover(parent_a: Wolf, parent_b: Wolf) -> Dict[str, int]:
    """Uniform crossover: for each layer, randomly pick from either parent."""
    schedule = {}
    for name in parent_a.rank_schedule:
        if random.random() < 0.5:
            schedule[name] = parent_a.rank_schedule[name]
        else:
            schedule[name] = parent_b.rank_schedule[name]
    return schedule


def mutate(schedule: Dict[str, int], diag_info, mutation_rate=0.4, mutation_scale=0.25):
    """Perturb ranks by ±scale fraction with given probability."""
    diag_map = {d["name"]: d for d in diag_info}
    new_schedule = {}
    for name, rank in schedule.items():
        if random.random() < mutation_rate:
            d = diag_map[name]
            delta = int(rank * mutation_scale * random.choice([-1, 1]))
            new_rank = rank + delta
            new_rank = max(4, min(new_rank, d["full_rank"]))
            new_schedule[name] = new_rank
        else:
            new_schedule[name] = rank
    return new_schedule


print(f"Layer names: {len(layer_names)}")
print(f"Diagnostics: {len(diag)}")
print("Wolf infrastructure ready.")

In [ ]:
# 11. Wolves evolutionary compression loop

# More aggressive hyperparams than JuliaFluxGPT run:
# - smaller pop (8 vs 10) since model is smaller
# - wider rank range (0.15-0.85 vs 0.3-1.0) to explore deep cuts
# - higher mutation (0.4/0.25 vs 0.3/0.15) for faster exploration
# - more gens (30 vs 25) since each is ~5x faster
POP_SIZE = 8
GENERATIONS = 30
CHILDREN_PER_GEN = 3
TOURNAMENT_K = 3
BETA = 0.01
MUTATION_RATE = 0.4
MUTATION_SCALE = 0.25
MIN_RANK_FRAC = 0.15
MAX_RANK_FRAC = 0.85

run = wandb.init(
    project="symbiogenesis",
    name="wolves-compress-juliaslm",
    config={
        "method": "wolves_svd_compression",
        "target_model": "JuliaSLM",
        "base_params": n_params,
        "base_loss": base_loss,
        "base_ppl": base_ppl,
        "base_entropy": base_entropy,
        "base_eff_rank": base_eff_rank,
        "architecture": "LLaMA-style MHA 4H",
        "d_model": config.d_model,
        "n_layers": config.n_layers,
        "pop_size": POP_SIZE,
        "generations": GENERATIONS,
        "children_per_gen": CHILDREN_PER_GEN,
        "tournament_k": TOURNAMENT_K,
        "beta": BETA,
        "mutation_rate": MUTATION_RATE,
        "mutation_scale": MUTATION_SCALE,
        "min_rank_frac": MIN_RANK_FRAC,
        "max_rank_frac": MAX_RANK_FRAC,
    },
    tags=["wolves", "compression", "svd", "juliaslm", "clean-weights"],
    reinit="finish_previous",
)

# Initialize population
print(f"Initializing population of {POP_SIZE} wolves...")
population = []
for i in range(POP_SIZE):
    schedule = random_rank_schedule(layer_names, diag, MIN_RANK_FRAC, MAX_RANK_FRAC)
    wolf = Wolf(rank_schedule=schedule)
    wolf = evaluate_wolf(wolf, model, val_inputs, val_labels, BETA)
    population.append(wolf)
    print(f"  Wolf {i}: {wolf.summary()}")

population.sort(key=lambda w: w.fitness, reverse=True)
print(f"\nBest initial: {population[0].summary()}")
print(f"Worst initial: {population[-1].summary()}")

# Evolution loop
t_start = time.time()
for gen in range(GENERATIONS):
    gen_start = time.time()
    replacements = 0

    for _ in range(CHILDREN_PER_GEN):
        parent_a = tournament_select(population, TOURNAMENT_K)
        parent_b = tournament_select(population, TOURNAMENT_K)
        child_schedule = crossover(parent_a, parent_b)
        child_schedule = mutate(child_schedule, diag, MUTATION_RATE, MUTATION_SCALE)
        child = Wolf(rank_schedule=child_schedule)
        child = evaluate_wolf(child, model, val_inputs, val_labels, BETA)

        tournament_idx = random.sample(range(len(population)), min(TOURNAMENT_K, len(population)))
        loser_idx = min(tournament_idx, key=lambda i: population[i].fitness)
        if child.fitness > population[loser_idx].fitness:
            population[loser_idx] = child
            replacements += 1

    population.sort(key=lambda w: w.fitness, reverse=True)
    best = population[0]
    mean_fitness = sum(w.fitness for w in population) / len(population)
    mean_params = sum(w.n_params for w in population) / len(population)
    gen_time = time.time() - gen_start

    wandb.log({
        "wolves/best_fitness": best.fitness,
        "wolves/best_loss": best.val_loss,
        "wolves/best_params": best.n_params,
        "wolves/best_entropy": best.weight_ent,
        "wolves/mean_fitness": mean_fitness,
        "wolves/mean_params": mean_params,
        "wolves/compression_ratio": 1 - best.n_params / n_params,
        "wolves/replacements": replacements,
        "wolves/generation": gen,
    }, step=gen)

    if gen % 5 == 0 or gen == GENERATIONS - 1:
        ppl = math.exp(min(best.val_loss, 20.0))
        print(f"Gen {gen:3d} | best: loss={best.val_loss:.4f} ppl={ppl:.1f} "
              f"params={best.n_params:,} ({1-best.n_params/n_params:.1%} smaller) | "
              f"replaced={replacements} | {gen_time:.1f}s")

elapsed = time.time() - t_start
print(f"\nEvolution complete in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Best wolf: {population[0].summary()}")

In [ ]:
# 12. Fine-tune the winning compressed model

best_wolf = population[0]
print(f"Best wolf: {best_wolf.summary()}")
print(f"\nRank schedule:")
for name, rank in sorted(best_wolf.rank_schedule.items()):
    d = {d_["name"]: d_ for d_ in diag}[name]
    print(f"  {name:<40} rank={rank:>4} / {d['full_rank']:<4} ({rank/d['full_rank']:.0%})")

# Create compressed model
compressed = compress_model(model, best_wolf.rank_schedule)
compressed.to(device)
compressed.train()

comp_params = sum(p.numel() for p in compressed.parameters())
print(f"\nCompressed JuliaSLM: {comp_params:,} params ({comp_params/1e6:.2f}M)")
print(f"Original:            {n_params:,} params ({n_params/1e6:.2f}M)")
print(f"Reduction:           {n_params - comp_params:,} ({(1-comp_params/n_params)*100:.1f}%)")

# Pre-finetune eval
pre_loss, pre_ppl = evaluate_model(compressed, val_inputs, val_labels)
print(f"\nPre-finetune: loss={pre_loss:.4f} ppl={pre_ppl:.1f}")

# Fine-tune
FINETUNE_STEPS = 2000
FINETUNE_LR = 6e-4  # matches JuliaSLM original training LR
FINETUNE_BS = 64
WARMUP = 100

optimizer = torch.optim.AdamW(
    compressed.parameters(), lr=FINETUNE_LR, weight_decay=0.1, betas=(0.9, 0.95)
)

def lr_lambda(step):
    if step < WARMUP:
        return (step + 1) / max(WARMUP, 1)
    progress = (step - WARMUP) / max(FINETUNE_STEPS - WARMUP, 1)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

n_train = len(train_inputs)
compressed.train()
t_start = time.time()
step = 0
best_ft_loss = float("inf")

print(f"\nFine-tuning compressed JuliaSLM for {FINETUNE_STEPS} steps...")
print(f"Precision: {amp_dtype}, batch={FINETUNE_BS}, lr={FINETUNE_LR}")

while step < FINETUNE_STEPS:
    perm = torch.randperm(n_train)
    for i in range(0, n_train, FINETUNE_BS):
        if step >= FINETUNE_STEPS:
            break

        idx = perm[i:i+FINETUNE_BS]
        batch_in = train_inputs[idx].to(device)
        batch_tgt = train_labels[idx].to(device)

        with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
            logits = compressed(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(compressed.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 50 == 0:
            wandb.log({"finetune/loss": loss.item(), "finetune/lr": scheduler.get_last_lr()[0]}, step=step)

        if step > 0 and step % 250 == 0:
            ft_loss, ft_ppl = evaluate_model(compressed, val_inputs, val_labels)
            wandb.log({"finetune/val_loss": ft_loss, "finetune/val_ppl": ft_ppl}, step=step)
            elapsed = time.time() - t_start
            marker = " ** NEW BEST **" if ft_loss < best_ft_loss else ""
            if ft_loss < best_ft_loss:
                best_ft_loss = ft_loss
            print(f"  [step {step:5d}] val_loss={ft_loss:.4f} ppl={ft_ppl:.1f} "
                  f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s){marker}")
            compressed.train()

        step += 1

# Final eval
post_loss, post_ppl = evaluate_model(compressed, val_inputs, val_labels)
post_entropy = weight_entropy(compressed)
elapsed = time.time() - t_start
print(f"\nFine-tune complete ({elapsed:.0f}s)")
print(f"Post-finetune: loss={post_loss:.4f} ppl={post_ppl:.1f}")
wandb.log({"finetune/final_loss": post_loss, "finetune/final_ppl": post_ppl})

In [ ]:
# 13. Final comparison

print("\n" + "=" * 60)
print("WOLVES COMPRESSION RESULTS — JuliaSLM")
print("=" * 60)
print(f"\n{'Metric':<20} {'Original':>12} {'Compressed':>12} {'Delta':>10}")
print("-" * 55)
print(f"{'Params':<20} {n_params:>12,} {comp_params:>12,} {comp_params-n_params:>+10,}")
print(f"{'Params (M)':<20} {n_params/1e6:>12.2f} {comp_params/1e6:>12.2f} {(comp_params-n_params)/1e6:>+10.2f}")
print(f"{'Val Loss':<20} {base_loss:>12.4f} {post_loss:>12.4f} {post_loss-base_loss:>+10.4f}")
print(f"{'Val PPL':<20} {base_ppl:>12.1f} {post_ppl:>12.1f} {post_ppl-base_ppl:>+10.1f}")
print(f"{'Weight Entropy':<20} {base_entropy:>12.4f} {post_entropy:>12.4f} {post_entropy-base_entropy:>+10.4f}")
print(f"{'Compression':<20} {'---':>12} {(1-comp_params/n_params)*100:>11.1f}% {'':>10}")

print(f"\nPre-finetune loss:  {pre_loss:.4f} (PPL {pre_ppl:.1f})")
print(f"Post-finetune loss: {post_loss:.4f} (PPL {post_ppl:.1f})")
print(f"Recovery:           {pre_ppl - post_ppl:.1f} PPL points")

# Compare to scaling law data points
print(f"\n{'='*60}")
print("SCALING LAW CONTEXT")
print(f"{'='*60}")
print(f"{'Model':<25} {'Params':>10} {'Val Loss':>10} {'Val PPL':>10}")
print("-" * 55)
print(f"{'JuliaFluxGPT-1M':<25} {'1.01M':>10} {'4.446':>10} {'85.3':>10}")
print(f"{'JuliaSLM-compressed':<25} {comp_params/1e6:>9.2f}M {post_loss:>10.4f} {post_ppl:>10.1f}")
print(f"{'SymbioSLM':<25} {'4.07M':>10} {'3.620':>10} {'37.3':>10}")
print(f"{'JuliaSLM (original)':<25} {'5.04M':>10} {'3.540':>10} {'34.5':>10}")
print(f"{'SymbioGPT-10M':<25} {'11.05M':>10} {'3.563':>10} {'35.3':>10}")

wandb.finish()

In [ ]:
# 14. Upload compressed model to HuggingFace
from huggingface_hub import HfApi, create_repo
import json

COMPRESSED_REPO = "LisaMegaWatts/JuliaSLM-compressed"

# Save compressed state dict
os.makedirs("compressed", exist_ok=True)
save_path = "compressed/juliaslm_compressed.pt"
torch.save(compressed.state_dict(), save_path)

# Save rank schedule + metadata
schedule_path = "compressed/rank_schedule.json"
with open(schedule_path, "w") as f:
    json.dump({
        "rank_schedule": best_wolf.rank_schedule,
        "source_model": "LisaMegaWatts/JuliaSLM",
        "architecture": "LLaMA-style MHA 4H + RMSNorm + SwiGLU + RoPE",
        "base_params": n_params,
        "compressed_params": comp_params,
        "base_loss": base_loss,
        "compressed_loss": post_loss,
        "base_ppl": base_ppl,
        "compressed_ppl": post_ppl,
        "compression_ratio": 1 - comp_params / n_params,
        "finetune_steps": FINETUNE_STEPS,
        "wolves_generations": GENERATIONS,
        "wolves_pop_size": POP_SIZE,
        "beta": BETA,
        "config": {
            "d_model": config.d_model,
            "n_layers": config.n_layers,
            "n_heads": config.n_heads,
            "head_dim": config.head_dim,
            "ffn_inner": config.ffn_inner,
            "context_length": config.context_length,
            "vocab_size": config.vocab_size,
        },
    }, f, indent=2)

print(f"Saved: {save_path} ({os.path.getsize(save_path)/1e6:.1f} MB)")
print(f"Saved: {schedule_path}")

# Upload to HF (new repo — never overwrite existing)
hf_api = HfApi()
try:
    create_repo(COMPRESSED_REPO, exist_ok=True)
    hf_api.upload_file(
        path_or_fileobj=save_path, path_in_repo="juliaslm_compressed.pt",
        repo_id=COMPRESSED_REPO,
        commit_message=f"Wolves-compressed JuliaSLM ({comp_params/1e6:.1f}M params, PPL={post_ppl:.1f})"
    )
    hf_api.upload_file(
        path_or_fileobj=schedule_path, path_in_repo="rank_schedule.json",
        repo_id=COMPRESSED_REPO,
        commit_message="Rank schedule and compression metadata"
    )
    print(f"\nUploaded to: https://huggingface.co/{COMPRESSED_REPO}")
except Exception as e:
    print(f"HF upload failed: {e}")

print("\nDone!")